In [1]:
import time
import pandas as pd
import numpy as np
import requests
from predict import predict

API_KEY = "IRJW2AFJ"
BASE    = "http://localhost:9999/v1"
HEADERS = {"X-API-Key": API_KEY}

# --- Config ---
TICKERS            = ["CRZY", "TAME"]
MAX_POSITION       = 10000
TRADE_SIZE         = 500
START_TRADING_TICK = 101
STOP_TRADING_TICK  = 290
MIN_ROWS_TO_PREDICT = 20  # need at least 20 rows for rolling windows

def get_case():
    return requests.get(f"{BASE}/case", headers=HEADERS).json()

def get_securities():
    return requests.get(f"{BASE}/securities", headers=HEADERS).json()

def get_book(ticker):
    return requests.get(f"{BASE}/securities/book",
                        params={"ticker": ticker}, headers=HEADERS).json()

def place_order(ticker, action, qty, price, order_type="LIMIT"):
    return requests.post(f"{BASE}/orders", headers=HEADERS, params={
        "ticker": ticker, "type": order_type,
        "quantity": qty, "action": action, "price": price
    }).json()

def cancel_all():
    try:
        requests.post(f"{BASE}/commands/cancel",
                      params={"all": 1}, headers=HEADERS)
    except Exception as e:
        print(f"Cancel failed: {e}")

def get_position(ticker):
    for s in get_securities():
        if s["ticker"] == ticker:
            return s["position"]
    return 0

def get_best_bid_ask(ticker):
    book = get_book(ticker)
    best_bid = book["bids"][0]["price"] if book["bids"] else None
    best_ask = book["asks"][0]["price"] if book["asks"] else None
    return best_bid, best_ask

def flatten_position(ticker):
    position = get_position(ticker)
    if position == 0:
        return
    qty = min(abs(int(position)), 25000)
    action = "SELL" if position > 0 else "BUY"
    print(f"[FLATTEN] {action} {qty} {ticker}")
    try:
        place_order(ticker, action, qty,
                    get_best_bid_ask(ticker)[0] if action == "SELL"
                    else get_best_bid_ask(ticker)[1],
                    order_type="MARKET")
    except Exception as e:
        print(f"Flatten error: {e}")

def snapshot_tick(tick):
    """Grab current market data for all tickers at this tick."""
    rows = []
    for s in get_securities():
        rows.append({
            "tick":       tick,
            "ticker":     s["ticker"],
            "last":       s["last"],
            "bid":        s["bid"],
            "ask":        s["ask"],
            "bid_size":   s["bid_size"],
            "ask_size":   s["ask_size"],
            "volume":     s["volume"],
            "position":   s["position"],
            "unrealized": s["unrealized"],
        })
    return rows

def execute_signal(signal_data):
    ticker    = signal_data["ticker"]
    signal    = signal_data["signal"]
    predicted = signal_data["predicted"]
    position  = get_position(ticker)
    best_bid, best_ask = get_best_bid_ask(ticker)

    if signal == "BUY" and position < MAX_POSITION:
        if best_ask is None:
            return
        qty = min(TRADE_SIZE, MAX_POSITION - int(position))
        print(f"[BUY]  {ticker} x{qty} @ {best_ask:.2f} | predicted {predicted:.2f}")
        try:
            place_order(ticker, "BUY", qty, best_ask)
        except Exception as e:
            print(f"Order error: {e}")

    elif signal == "SELL" and position > -MAX_POSITION:
        if best_bid is None:
            return
        qty = min(TRADE_SIZE, MAX_POSITION + int(position))
        print(f"[SELL] {ticker} x{qty} @ {best_bid:.2f} | predicted {predicted:.2f}")
        try:
            place_order(ticker, "SELL", qty, best_bid)
        except Exception as e:
            print(f"Order error: {e}")

    else:
        print(f"[HOLD] {ticker} | position {position}")


def run_trading():
    # Load the CSV saved by collect.py as starting point
    live_df   = pd.read_csv("rit_first100.csv")
    last_tick = -1
    signals   = {}

    # Generate first predictions from collected data
    for ticker in TICKERS:
        ticker_rows = live_df[live_df["ticker"] == ticker]
        if len(ticker_rows) >= MIN_ROWS_TO_PREDICT:
            signals[ticker] = predict(live_df, ticker=ticker)
            print(f"Initial signal {ticker}: {signals[ticker]['signal']}")
        else:
            print(f"Not enough data for {ticker} yet")

    print("\n=== TRADING STARTED ===")

    while True:
        case = get_case()

        if case["status"] != "ACTIVE":
            print("Case no longer active. Exiting.")
            break

        tick = case["tick"]

        # Skip duplicate ticks
        if tick == last_tick:
            time.sleep(0.2)
            continue
        last_tick = tick

        # Flatten and exit near end of case
        if tick >= STOP_TRADING_TICK:
            print(f"\n=== Tick {tick}: Flattening all positions ===")
            cancel_all()
            for ticker in TICKERS:
                flatten_position(ticker)
            break

        # Only trade after collection window
        if tick < START_TRADING_TICK:
            time.sleep(0.2)
            continue

        # --- Append live snapshot to growing dataframe ---
        new_rows = snapshot_tick(tick)
        live_df  = pd.concat([live_df, pd.DataFrame(new_rows)],
                              ignore_index=True)

        print(f"\n--- Tick {tick} | live_df rows: {len(live_df)} ---")

        # Refresh predictions every 20 ticks using the growing dataframe
        if tick % 20 == 0:
            print("Refreshing predictions...")
            for ticker in TICKERS:
                ticker_rows = live_df[live_df["ticker"] == ticker]
                if len(ticker_rows) >= MIN_ROWS_TO_PREDICT:
                    signals[ticker] = predict(live_df, ticker=ticker)
                    print(f"  {ticker}: {signals[ticker]['signal']} "
                          f"| predicted {signals[ticker]['predicted']:.2f}")
                else:
                    print(f"  {ticker}: not enough data yet "
                          f"({len(ticker_rows)} rows)")

        # Execute signals
        cancel_all()
        for ticker in TICKERS:
            if ticker in signals:
                execute_signal(signals[ticker])

        time.sleep(0.5)

    print("\n=== TRADING COMPLETE ===")
    for ticker in TICKERS:
        print(f"Final position {ticker}: {get_position(ticker)}")

    # Save full session data for later analysis
    live_df.to_csv("rit_full_session.csv", index=False)
    print("Full session saved to rit_full_session.csv")


if __name__ == "__main__":
    run_trading()

Not enough data for CRZY yet
Not enough data for TAME yet

=== TRADING STARTED ===

--- Tick 101 | live_df rows: 4 ---

--- Tick 102 | live_df rows: 6 ---

--- Tick 103 | live_df rows: 8 ---

--- Tick 104 | live_df rows: 10 ---

--- Tick 105 | live_df rows: 12 ---

--- Tick 106 | live_df rows: 14 ---

--- Tick 107 | live_df rows: 16 ---

--- Tick 108 | live_df rows: 18 ---

--- Tick 109 | live_df rows: 20 ---

--- Tick 110 | live_df rows: 22 ---

--- Tick 111 | live_df rows: 24 ---

--- Tick 112 | live_df rows: 26 ---

--- Tick 113 | live_df rows: 28 ---

--- Tick 114 | live_df rows: 30 ---

--- Tick 115 | live_df rows: 32 ---

--- Tick 116 | live_df rows: 34 ---

--- Tick 117 | live_df rows: 36 ---

--- Tick 118 | live_df rows: 38 ---

--- Tick 119 | live_df rows: 40 ---

--- Tick 120 | live_df rows: 42 ---
Refreshing predictions...

=== CRZY PREDICTION ===
Current price:    9.12
Predicted next:   7.55
MA5:              9.43
MA20:             9.22
Momentum:         -0.0600
Volatility: